In [ ]:
import pandas as pd
import xgboost as xgb
import joblib
from sklearn.metrics import accuracy_score, classification_report

# 1. Load the Data
# train_data = pd.read_csv('testing_dataset\02_testing.csv')

train_data = pd.read_csv('training_dataset\00_training.csv')
# test_data = pd.read_csv('training_dataset\00_training.csv')

test_data = pd.read_csv('testing_dataset\01_validation.csv')

# 2. Separate the "Features" (X) from the "Answers" (y)
# Drop the LABEL column to create the features, keep it for the target
X_train = train_data.drop('LABEL', axis=1)
y_train = train_data['LABEL']

# Do the exact same for the Final Exam (Testing Data)
X_test = test_data.drop('LABEL', axis=1)
y_test = test_data['LABEL'] # <- This is your Answer Key!

# 3. Create and Train the XGBoost Model (The Studying Phase)
print("Training the model... (This might take a minute)")
model = xgb.XGBClassifier(
    n_estimators=100,      # How many decision trees to build
    learning_rate=0.1,     # How fast the model learns
    random_state=42        # Ensures we get the same result every time
)
model.fit(X_train, y_train)

# 4. The Final Exam (Testing Phase)
# We ONLY give it X_test (the features). We hide y_test (the answers).
print("Taking the Final Exam...")
predictions = model.predict(X_test)

# 5. Grade the Exam!
# We compare the AI's guesses (predictions) against the true answers (y_test)
accuracy = accuracy_score(y_test, predictions)
print(f"\nModel Accuracy: {accuracy * 100:.2f}%")

# Save the model with joblib
joblib.dump(model, 'model_v1.joblib')
print("Model saved as 'model_v1.joblib'!")

# Print a detailed report card showing how well it did on 0, 1, and 2
print("\nDetailed Report Card:")
print(classification_report(y_test, predictions))

In [ ]:
import pandas as pd
import xgboost as xgb
import joblib
from sklearn.metrics import accuracy_score

train_df = pd.read_csv('training_dataset\00_training.csv')
val_df = pd.read_csv('testing_dataset\01_validation.csv')
test_df = pd.read_csv('testing_dataset\02_testing.csv')
external_df = pd.read_csv('testing_dataset\03_external_test_set.csv')

X_train, y_train = train_df.drop('LABEL', axis=1), train_df['LABEL']
X_val, y_val = val_df.drop('LABEL', axis=1), val_df['LABEL']
X_test, y_test = test_df.drop('LABEL', axis=1), test_df['LABEL']
X_ext, y_ext = external_df.drop('LABEL', axis=1), external_df['LABEL']

print("Training the balanced model...")
model = xgb.XGBClassifier(
    n_estimators=800,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_lambda=0.1,
    reg_alpha=0.0,
    early_stopping_rounds=50
)

model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
)

try:
    model.save_model("drug_food_model.json")
    print("SUCCESS! 'drug_food_model.json' has been created and saved!")
except FileNotFoundError:
    print(" Error: Please make sure '01_validation.csv' is uploaded to Colab first.")

# Save the model with joblib
joblib.dump(model, 'drug_food_model.joblib')
print("Model saved as 'drug_food_model.joblib'!")

print("\n--- Model Evaluation ---")

test_predictions = model.predict(X_test)
test_acc = accuracy_score(y_test, test_predictions)
print(f"Testing Accuracy (Internal): {test_acc * 100:.2f}%")

ext_predictions = model.predict(X_ext)
ext_acc = accuracy_score(y_ext, ext_predictions)
print(f"External Accuracy (Real-World): {ext_acc * 100:.2f}%")